In [ ]:
# Stastics - 
# Raw numbers alone tells nothing. Statistics tells what is normal, what is unusual, and how confident you can be. 
# Every alert threshold, every anomaly score, every SLA calculation is built on basic statistics.

In [2]:
# Measurs of central tendency - where is the middle ?

import numpy as np
cpu = np.array([42, 55, 78, 91, 63, 78, 45, 78, 91, 55])
print(np.mean(cpu))  #average
print(np.median(cpu)) # middle value when sorted
# Mean is pulled by extreme values. If one server spikes to 99%, the mean rises even if everything else is fine. 
# Median ignores extremes — it is the middle value when sorted. 
# In AIOps use median when you have spikes you don't want distorting your baseline.

67.6
70.5


In [3]:
# Measures of spread - how variable are readings ?
import numpy as np
cpu = np.array([42, 55, 78, 91, 63, 78, 45, 78, 91, 55])

print(np.std(cpu))  # average distance from mean
print(np.var(cpu))  # std squared
print(np.ptp(cpu))  # peak to peak, max minus min

17.10087717048456
292.44
49


In [ ]:
# low std means stable and predictable.
# High std means erratic - investigate
# ptp (peak to peak) gives the full range in one number useful for spotting servers with wild swings.

In [3]:
#percentiles - where does a value rank ?

cpu = np.array([42,55,78,91,63,78,45,78,91,55])
print(np.percentile(cpu, 50))
print(np.percentile(cpu, 90))
print(np.percentile(cpu, 95))
print(np.percentile(cpu, 99))

70.5
91.0
91.0
91.0


In [ ]:
# Percentiles are how SLAs are defined in real systems. 
# "p99 latency below 200ms" means 99% of requests must complete within 200ms. 
# The 99th percentile is the threshold you care about — not the average.

In [5]:
#p95 and p99 thresholds - production alerting

import numpy as np
latency = np.array ([12, 15, 14, 13, 16, 14, 15, 45, 13, 14, 15, 12, 60, 14, 13, 15])
p95 = np.percentile(latency, 95)
p99 = np.percentile(latency, 99)
print(f"p95 threshold: {p95:.1f} ms")
print(f"p99 threshold: {p99:.1f} ms")
#How many requests breached p95 
breaches = np.sum(latency > p95)
breach_pct = np.mean(latency > p95) * 100
print(f"p95 breaches: {breaches}")
print(f"breach rate : {breach_pct:.1f} ")

p95 threshold: 48.8 ms
p99 threshold: 57.7 ms
p95 breaches: 1
breach rate : 6.2 


In [ ]:
# This is exactly how an SLA report is built. You calculate p95 or p99 from historical data, 
# then count how many readings exceeded it.

In [6]:
# CORELATION - DO TWO METRICS MOVE TOGETHER ??
cpu = np.array([42,55,78,91,63,30,35,88])
latency = np.array([14,18,35,55,22,12,13,48])

#correlation matrix - values between -1 and +1
corr = np.corrcoef(cpu, latency)
print(corr.round(2))

#the actual correlation value

print(f"correlation: {corr[0,1]:.2f}")

[[1.   0.95]
 [0.95 1.  ]]
correlation: 0.95


In [ ]:
# Correlation of 0.99 means CPU and latency rise and fall almost perfectly together. 
# In AIOps this tells you CPU is likely causing the latency spikes — a strong lead for root cause analysis. 
# Values close to 0 mean no relationship. Negative values mean one goes up as the other goes down.

In [5]:
# SLIDING WINDOW CORRELATION
import numpy as np

cpu = np.array([42,55,78,91,63,30,35,88])
latency = np.array([14,18,35,55,22,12,13,48])

window = 3  # number of points per window

for i in range(len(cpu) - window + 1):
    cpu_window = cpu[i:i+window]
    lat_window = latency[i:i+window]

    corr = np.corrcoef(cpu_window, lat_window)[0,1]

    print(f"Index {i} to {i+window-1} → correlation: {round(corr,2)}")

    if corr > 0.8:
        print(f"**Strong correlation at window {i}-{i+window-1}")

Index 0 to 2 → correlation: 0.98
**Strong correlation at window 0-2
Index 1 to 3 → correlation: 0.98
**Strong correlation at window 1-3
Index 2 to 4 → correlation: 0.99
**Strong correlation at window 2-4
Index 3 to 5 → correlation: 0.94
**Strong correlation at window 3-5
Index 4 to 6 → correlation: 1.0
**Strong correlation at window 4-6
Index 5 to 7 → correlation: 1.0
**Strong correlation at window 5-7


In [4]:
# MOVING AVERAGE - SMOOTHING OUT NOISE
import numpy as np

cpu = np.array([60,62,58,91,61,63,59,60,95,62])
window = 3
# calculate 3-point moving average
moving_avg = np.array([
    cpu[i:i + window].mean()
    for i in range(len(cpu) - window + 1)
])
# detect deviation   
threshold = np.mean(moving_avg) + 2*np.std(moving_avg)

anomalies = moving_avg > threshold

print("Moving Avg:", moving_avg.round(1))
print("Anomalies :", anomalies)

anomaly_indices = np.where(anomalies)[0]

# map back to original index
real_indices = anomaly_indices + window - 1

print("Incident timestamps:", real_indices)

Moving Avg: [60.  70.3 70.  71.7 61.  60.7 71.3 72.3]
Anomalies : [False False False False False False False False]
Incident timestamps: []


In [ ]:
# Raw readings jump around. Moving average smooths the noise so trends become visible. 
# A sudden rise in the moving average is more reliable than a single spike — it means the problem is sustained, not a one-off blip.

In [10]:
#ALL TOGETHER
import numpy as np
servers = np.array([
    [55, 60, 78, 91, 85], #server A
    [30, 32, 35, 33, 31], #server B
    [70, 75, 88, 95, 99], #server C
])

print("mean per server: ", servers.mean(axis=1).round(1))
print("std per server:", servers.std(axis=1).round(1))
print("median per server", np.median(servers, axis=1))
print("p95 per server", np.percentile(servers, 95, axis=1))
print("max per server", servers.max(axis=1))

mean per server:  [73.8 32.2 85.4]
std per server: [14.   1.7 11.2]
median per server [78. 32. 88.]
p95 per server [89.8 34.6 98.2]
max per server [91 35 99]


In [ ]:
# observation: Server B has std of 1.7 — very stable. Server C has std of 10.7 — much more variable. 
# Server A mean is 73.8 but p95 is 89.4 — it spikes high occasionally even though average looks acceptable.

In [38]:
# Dataset 
import numpy as np
cpu = np.array([60,62,58,61,63,59,91,60,62,58,95,61,60,63,59,60])
latency =  np.array([12,14,13,15,14,12,55,13,14,12,60,14,13,15,12,13])

servers = np.array([
    [55, 60, 78, 91, 85],
    [30, 32, 35, 33, 31],
    [70, 75, 88, 95, 99],
])

print("mean", np.mean(cpu))
print("median", np.median(cpu))
print("standard deviation", np.std(cpu))
print("peak to peak", np.ptp(cpu))
p95 = np.percentile(latency, 95)
p99 = np.percentile(latency, 99)
print(f"p95: {p95:.1f}ms")
print(f"p99: {p99:.1f}ms")
percentage = np.sum(latency > p95) * 100 / len(latency)
print(f"Percentage of latency readings exceeding p95: {percentage}")
print(f"correlation between cpu and latency")
corr = np.corrcoef(cpu, latency)
print(corr.round(2))
print(f"correlation: {corr[0,1]:.2f}")
print(" mean, std, median and p95 per server")
print("mean per server", servers.mean(axis=1).round(1))
print("std per server", servers.std(axis=1).round(1) )
print("median per server", np.median(servers, axis=1).round(1))
print("p95 per server", np.percentile(servers, 95, axis=1).round(1))
print("max per server", servers.max(axis=1))

mean 64.5
median 60.5
standard deviation 10.897247358851684
peak to peak 37
p95: 56.2ms
p99: 59.2ms
Percentage of latency readings exceeding p95: 6.25
correlation between cpu and latency
[[1. 1.]
 [1. 1.]]
correlation: 1.00
 mean, std, median and p95 per server
mean per server [73.8 32.2 85.4]
std per server [14.   1.7 11.2]
median per server [78. 32. 88.]
p95 per server [89.8 34.6 98.2]
max per server [91 35 99]
